This notebook plots summary staistics by each predicted ancestry group after removing non-monomorphic variants in the respective population

In [ ]:
requiredPackages = c('tidyverse', 'ggsci', 'data.table')
for(p in requiredPackages){
  if(!require(p,character.only = TRUE)) install.packages(p)
  library(p,character.only = TRUE)
}

In [ ]:
colorBlindBlack8  <- c("#000000", "#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7")

# Copy summary statistics to persistent disk

In [ ]:
setwd("/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility")

In [ ]:
ancestry_pred <- fread("./meta/ancestry_preds.tsv") %>%
    select(all_of(c("research_id", "ancestry_pred_other")))

head(ancestry_pred)

In [ ]:
if (!dir.exists("/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/plink_array_sumstats_bygroup")){

    dir.create("/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/plink_array_sumstats_bygroup")

    my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

    for (anc in unique(ancestry_pred$ancestry_pred_other)) {

        system(paste0("gsutil cp ", my_bucket,
                      "/data/meta/plink_array/sumstats_", anc, "/* ",
                      "/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/plink_array_sumstats_bygroup/"),
               intern=T)

    }


    
    list.files(path = "/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/plink_array_sumstats_bygroup/",
               pattern = NULL,
               all.files = FALSE,
               full.names = FALSE)
    
}

# Script to get combined table of summary statistics

In [ ]:
get_sumstats <- function(
    sum.dir = "/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/plink_array_sumstats_bygroup",
    file.extension, # Write hwe not .hwe
    select.cols,
    file.prefix = "plinkarray_sumstats_"
){
    # Loads the file, makes a list of dfs and names the df by ancestry
    file_names <- list.files(paste0(sum.dir, "/"), pattern = paste0("\\.", file.extension, "$"))
    list_df <- lapply(paste0(sum.dir, "/", file_names), fread)
    names(list_df) <- gsub(paste0(file.prefix, "(.*)\\.", file.extension), "\\1", file_names)
    
    # Select columns
    list_df <- lapply(list_df, function(df) df %>% select(all_of(select.cols)))
                      
    # Adds an ancestry column to it
    for (name in names(list_df)) {
        list_df[[name]]$ancestry <- name
    }
    
    # Concatanates dataaframes
    list_df <- do.call(rbind, list_df)
    
    return(list_df)
}

# MAF

In [ ]:
maf <- get_sumstats(file.extension = "frq", select.cols = c("CHR", "SNP", "MAF"))

In [ ]:
table(is.na(maf$MAF))

In [ ]:
maf <- maf %>%
    filter(!is.na(MAF)) %>%
    mutate(pass5 = ifelse(MAF > 0.05, 1, 0)) %>%
    mutate(pass1 = ifelse(MAF > 0.01, 1, 0)) %>%
    group_by(ancestry) %>%
    mutate(frac5 = sum(pass5)/n(),
           frac1 = sum(pass1)/n(),
           label = paste0(ancestry, " (5%: ", round(frac5*100, digits = 1), "%, ",
                          "1%: ", round(frac1*100, digits = 1), "%)"))

dim(maf)
head(maf)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_maf <- ggplot(maf, aes(x = MAF))+
    geom_histogram(aes(fill = ancestry), binwidth = 0.05)+
    xlab("Minor allele frequency")+
    ylab("Number of variants")+
    ggtitle("Distribution of MAF in AoU array data by\npredicted ancestry (polymorphic variants")+
    #scale_x_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))+
    theme_bw()+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "none")+
    facet_wrap(~label, ncol = 2, scales = 'free_y')

p_maf

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_maf+
    scale_x_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))

# Hardy-Weinberg equilibrium

In [ ]:
hwe <- get_sumstats(file.extension = "hwe", select.cols = c("CHR", "SNP", "P")) %>%
    mutate(logP = -log10(P))

In [ ]:
hwe <- hwe %>%
    filter(!is.na(P)) %>%
    mutate(pass7 = ifelse(logP < 7, 1, 0)) %>%
    mutate(pass12 = ifelse(logP < 12, 1, 0)) %>%
    group_by(ancestry) %>%
    mutate(frac7 = sum(pass7)/n(),
           frac12 = sum(pass12)/n(),
           label = paste0(ancestry, " (10e-7: ", round(frac7*100, digits = 1),
                          "%, 10e-12: ", round(frac12*100, digits = 1), "%)"))

dim(hwe)
head(hwe)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_hwe <- ggplot(hwe, aes(x = logP))+
    geom_histogram(aes(fill = ancestry), binwidth = 10)+
    xlab("Hardy-Weinberg equilibrium -log10(p)")+
    ylab("Number of variants")+
    ggtitle("Distribution of HWE p-value in AoU array data\nby predicted ancestry (polymorphic variants)")+
    #guides(fill = guide_legend(title = "Predicted ancestry", nrow = 1))+
    theme_bw()+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "none")+
    facet_wrap(~label, ncol = 2, scales = 'free_y')

p_hwe

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_hwe+
    scale_y_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))

# Heterozygosity

Here we use plink's --het output to find the heterozygosity rate per individual. The mean heterozygosity rate per individual is defined as follows: (N(NM) - O(HM)) / N(NM), where N(NM) is the rate of nonmissing genotypes per individual and O(HM) is the observed rate of homozygous genotypes per individual. The distribution of heterozygosity per individual is dataset dependent, and heterozygosity thresholds should therefore be selected in a cohort specific manner. We suggest visualizing the distribution and selecting thresholds based on the corresponding mean and standard deviation.

In [ ]:
het <- get_sumstats(file.extension = "het", select.cols = c("IID", "O(HOM)", "N(NM)"))

In [ ]:
het <- het %>%
    mutate(het_rate = (`N(NM)` - `O(HOM)`)/`N(NM)`) %>%
    group_by(ancestry) %>%
    mutate(lim_lo = mean(het_rate) - 3*sd(het_rate),
           lim_hi = mean(het_rate) + 3*sd(het_rate),
           pass = ifelse((het_rate > lim_lo) & (het_rate < lim_hi), 1, 0),
           frac = sum(pass)/n(),
           label = paste0(ancestry, " (", round(frac*100, digits = 3), "%)"))

dim(het)
head(het)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_het <- ggplot(het, aes(x = het_rate))+
    geom_histogram(aes(fill = ancestry), binwidth = 0.0005)+
    xlab("Heterozygosity")+
    ylab("Number of individuals")+
    geom_vline(aes(xintercept = lim_lo, group = ancestry), linetype = 2)+
    geom_vline(aes(xintercept = lim_hi, group = ancestry), linetype = 2)+
    ggtitle("Distribution of heterozygosity in in AoU array data by predicted\nancestry (polymorphic variants) (Dotted line: mean +/- 3*SD)")+
    #guides(fill = guide_legend(title = "Predicted ancestry", nrow = 1))+
    theme_bw()+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "none")+
    facet_wrap(~label, ncol = 2, scales = 'free_y')

p_het

In [ ]:
# Saving the .remove files for heterozygosity

my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

for (anc in unique(het$ancestry)){
    
    het_sub <- het %>%
        ungroup() %>%
        filter((pass == 0) & (ancestry == anc)) %>%
        mutate(FID = 0) %>%
        select(all_of(c("FID", "IID")))

    write.table(het_sub,
                file = paste0("./meta/plink_array_het_", anc, ".remove"),
                quote=FALSE, sep='\t', row.names = FALSE, col.names = TRUE)
    
    system(paste0("gsutil cp ./meta/plink_array_het_", anc, ".remove ",
                  my_bucket, "/data/meta/plink_array/remove_het_outliers/"),
           intern=T)
}

system(paste0("gsutil ls ", my_bucket, "/data/meta/plink_array/remove_het_outliers/"), intern=T)

# Individual missingness

In [ ]:
imiss <- get_sumstats(file.extension = "imiss", select.cols = c("IID", "F_MISS"))

dim(imiss)
head(imiss)

In [ ]:
table(imiss$ancestry)
100*100/(123062*2)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_imiss <- ggplot(imiss, aes(x = F_MISS))+
    geom_histogram(aes(fill = ancestry), binwidth = 0.00025)+
    xlab("Proportion of missing variants for individuals")+
    ylab("Number of individuals")+
    ggtitle("Distribution of individual missingness in AoU array data\nby predicted ancestry (polymorphic variants)")+
    #guides(fill = guide_legend(title = "Predicted ancestry", nrow = 1))+
    theme_bw()+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "none")+
    facet_wrap(~ancestry, ncol = 2, scales = 'free_y')

p_imiss

# Variant missingness

In [ ]:
lmiss <- get_sumstats(file.extension = "lmiss", select.cols = c("CHR", "SNP", "F_MISS"))

In [ ]:
lmiss <- lmiss %>%
    filter(!is.na(F_MISS)) %>%
    mutate(pass5 = ifelse(F_MISS < 0.05, 1, 0)) %>%
    mutate(pass1 = ifelse(F_MISS < 0.1, 1, 0)) %>%
    group_by(ancestry) %>%
    mutate(frac5 = sum(pass5)/n(),
           frac1 = sum(pass1)/n(),
           label = paste0(ancestry, " (0.05: ", round(frac5*100, digits = 2), "%, ",
                          "0.1: ", round(frac1*100, digits = 2), "%)"))

dim(lmiss)
head(lmiss)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_lmiss <- ggplot(lmiss, aes(x = F_MISS))+
    geom_histogram(aes(fill = ancestry), binwidth = 0.0075)+
    xlab("Proportion of missing individuals for variants")+
    ylab("Number of variants")+
    ggtitle("Distribution of variant missingness in AoU array data\nby predicted ancestry  (polymorphic variants)")+
    #guides(fill = guide_legend(title = "Predicted ancestry", nrow = 1))+
    #geom_vline(xintercept = 0.1, linetype = 2)+
    theme_bw()+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "none")+
    facet_wrap(~label, ncol = 2, scales = 'free_y')

p_lmiss

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

p_lmiss+
    scale_y_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))

# Density plots

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 7)

p_maf_density <- ggplot(maf, aes(x = MAF, color = ancestry))+
    geom_density(linewidth = 1)+
    xlab("Minor allele frequency")+
    ylab("Density")+
    ggtitle("Distribution of MAF in AoU array data\n(polymorphic variants)")+
    #scale_x_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))+
    guides(color = guide_legend(title = "Predicted ancestry", nrow = 2))+
    theme_bw()+
    scale_color_manual(values = colorBlindBlack8)+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "top")

ggsave(
    "./fig/maf.eps",
    plot = p_maf_density,
    device = "eps",
    width = 7,
    height = 7
)

p_maf_density

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 7)

p_het_density <- ggplot(het, aes(x = het_rate, color = ancestry))+
    geom_density(linewidth = 1)+
    xlab("Heterozygosity")+
    ylab("Density")+
    ggtitle("Distribution of heterozygosity\nin AoU array data (polymorphic variants)")+
    #scale_x_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))+
    guides(color = guide_legend(title = "Predicted ancestry", nrow = 2))+
    theme_bw()+
    scale_color_manual(values = colorBlindBlack8)+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "top")

p_het_density

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 7)

p_hwe_density <- ggplot(hwe, aes(x = logP, color = ancestry))+
    geom_density(linewidth = 1)+
    xlab("Hardy-Weinberg equilibrium -log10(p)")+
    ylab("Density")+
    ggtitle("Distribution of HWE p-value\nin AoU array data (polymorphic variants)")+
    #scale_x_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))+
    guides(color = guide_legend(title = "Predicted ancestry", nrow = 2))+
    theme_bw()+
    scale_color_manual(values = colorBlindBlack8)+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "top")

ggsave(
    "./fig/hwe.eps",
    plot = p_hwe_density,
    device = "eps",
    width = 7,
    height = 7
)

p_hwe_density

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 7)

p_imiss_density <- ggplot(imiss, aes(x = F_MISS, color = ancestry))+
    geom_density(linewidth = 1)+
    xlab("Proportion of missing variants for individuals")+
    ylab("Density")+
    ggtitle("Distribution of individual missingness\nin AoU array data (polymorphic variants)")+
    #scale_x_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))+
    guides(color = guide_legend(title = "Predicted ancestry", nrow = 2))+
    theme_bw()+
    scale_color_manual(values = colorBlindBlack8)+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "top")

ggsave(
    "./fig/imiss.eps",
    plot = p_imiss_density,
    device = "eps",
    width = 7,
    height = 7
)

p_imiss_density

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 7)

p_lmiss_density <- ggplot(lmiss, aes(x = F_MISS, color = ancestry))+
    geom_density(linewidth = 1)+
    xlab("Proportion of missing individuals for variants")+
    ylab("Density")+
    ggtitle("Distribution of variant missingness\nin AoU array data (polymorphic variants)")+
    #scale_x_log10(breaks = 10^(-10:10), minor_breaks = rep(1:9, 21)*(10^rep(-10:10, each=9)))+
    guides(color = guide_legend(title = "Predicted ancestry", nrow = 2))+
    theme_bw()+
    scale_color_manual(values = colorBlindBlack8)+
    theme(plot.title = element_text(hjust = 0.5),
          text = element_text(size = 16),
          legend.position = "top")

ggsave(
    "./fig/lmiss.eps",
    plot = p_lmiss_density,
    device = "eps",
    width = 7,
    height = 7
)

p_lmiss_density

In [ ]:
sessionInfo()